# 2. Mixed-Model Analysis of the Balanced Data; Hypothesis Tests; BLUE and BLUP

*Companion to Chapter 1, pages 12–23.*

The mixed model puts only fixed effects (here just $\mu$) in the mean and declares the other terms random, estimating their variances directly by restricted maximum likelihood (REML). This notebook mirrors the SAS PROC MIXED, lme4 and ASReml analyses with three Python engines:

* **statsmodels** `MixedLM` — the primary tool; crossed random effects are expressed as variance components inside one all-encompassing group;
* **lmmtools** `SparseREML` (custom, Appendix C.1) — for SAS-style standard errors (COVTEST), likelihood-ratio tests and profile confidence intervals, which `MixedLM` does not report for variance components;
* **pyreml** — shown in Appendix B as the alternative for larger or structured models.

In [1]:
# Modules and imports for this section
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf                # MixedLM via mixedlm()
from scipy import stats                              # chi-square for LRTs
from lmmtools.data import load_maize                 # CUSTOM: data loader (Appendix A)
from lmmtools.design import incidence                # CUSTOM: sparse incidence (Z) matrices
from lmmtools.reml import SparseREML, boundary_lrt   # CUSTOM: sparse REML, SEs, LRT, profile CIs (Appendix C.1)
from lmmtools.report import check
import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.width", 100)
maize = load_maize(balanced=True)

## Balanced data: mixed-model analysis with SAS PROC MIXED

**Original SAS code (from the chapter)**

```sas
proc mixed data=ril.maizeril covtest;
class location rep RIL;
model height = /solution;
random location rep(location) RIL RIL*location/solution;
ods output solutionR = random solutionF = fixed;
```

In `statsmodels`, `groups` must be a single column covering all observations (`one`), and each crossed random term becomes a variance-component formula. The default optimiser stops slightly short of the optimum on this flat likelihood (location has only four levels); BFGS with a tight gradient tolerance reaches the SAS solution <sup>[N4]</sup>.

> **Note N4** (statsmodels). statsmodels MixedLM's default optimiser stops at location = 221.01; method='bfgs' with gtol=1e-10 reaches SAS's 220.66.

In [2]:
d = maize.assign(one=1)
vc = {"location": "0 + C(location)", "rep(location)": "0 + C(location):C(rep)",
      "RIL": "0 + C(RIL)", "location*RIL": "0 + C(RIL):C(location)"}
mm = smf.mixedlm("height ~ 1", d, groups="one", vc_formula=vc).fit(reml=True, method=["bfgs"], gtol=1e-10, maxiter=5000)
sm_vc = pd.Series(dict(zip(mm.model.exog_vc.names, mm.vcomp)) | {"Residual": mm.scale})
print(sm_vc.round(4))
print(f"-2 Res Log Likelihood {-2 * mm.llf:.1f}   intercept {mm.fe_params.iloc[0]:.2f} (SE {mm.bse_fe.iloc[0]:.4f})")

RIL              303.1511
location         220.6614
location*RIL      24.9323
rep(location)     13.4463
Residual          64.8862
dtype: float64
-2 Res Log Likelihood 3833.2   intercept 182.12 (SE 7.8719)


In [3]:
for comp, val in [("location", 220.66), ("rep(location)", 13.4463), ("RIL", 303.15), ("location*RIL", 24.9323), ("Residual", 64.8862)]:
    check(f"statsmodels sigma2 {comp}", sm_vc[comp], val, 0.006 if comp in ("location", "RIL") else 5e-4)
check("-2 Res Log Likelihood", -2 * mm.llf, 3833.2, 0.05)

PASS  statsmodels sigma2 location: Python = 220.661   chapter = 220.66
PASS  statsmodels sigma2 rep(location): Python = 13.4463   chapter = 13.4463
PASS  statsmodels sigma2 RIL: Python = 303.151   chapter = 303.15
PASS  statsmodels sigma2 location*RIL: Python = 24.9323   chapter = 24.9323
PASS  statsmodels sigma2 Residual: Python = 64.8862   chapter = 64.8862
PASS  -2 Res Log Likelihood: Python = 3833.19   chapter = 3833.2


`MixedLM`'s summary lists "Var" rows with a standard error, but these are not the asymptotic SEs of the variance components that `covtest` prints. The custom `SparseREML` computes the exact observed-information SEs (as PROC MIXED) and the SAS fit statistics:

In [4]:
Zs = [incidence(maize, f)[0] for f in ["location", ["location", "rep"], "RIL", ["location", "RIL"]]]
names = ["location", "rep(location)", "RIL", "location*RIL"]
model = SparseREML(maize.height, np.ones((len(maize), 1)), Zs, names)
fit = model.fit()                        # REML; se="observed" (SAS COVTEST)
print(fit.table().round(4))
print(f"-2 Res Log Likelihood {fit.neg2_loglik:.1f}   AIC {fit.aic:.1f}")
print(f"Solution for fixed effects: Intercept {fit.beta[0]:.2f}, SE {fit.beta_se[0]:.4f}, DF 3, t {fit.beta[0] / fit.beta_se[0]:.2f}")

               Estimate  Std Error  Z value    Pr Z
location       220.6615   186.4848   1.1833  0.1184
rep(location)   13.4463    10.2484   1.3120  0.0948
RIL            303.1515    57.5090   5.2714  0.0000
location*RIL    24.9323     6.6787   3.7331  0.0001
Residual        64.8862     5.8745  11.0454  0.0000
-2 Res Log Likelihood 3833.2   AIC 3843.2
Solution for fixed effects: Intercept 182.12, SE 7.8719, DF 3, t 23.14


In [5]:
sas = pd.DataFrame({"Estimate": [220.66, 13.4463, 303.15, 24.9323, 64.8862],
                    "Std Error": [186.48, 10.2484, 57.5088, 6.6787, 5.8745]}, index=fit.table().index)
for comp in sas.index:
    check(f"sigma2 {comp}", fit.table().loc[comp, "Estimate"], sas.loc[comp, "Estimate"], 2e-4 * sas.loc[comp, "Estimate"])
    check(f"SE {comp}", fit.table().loc[comp, "Std Error"], sas.loc[comp, "Std Error"], 2e-4 * sas.loc[comp, "Std Error"])
check("AIC (SAS)", fit.aic, 3843.2, 0.05)
check("Intercept SE", fit.beta_se[0], 7.8719, 5e-5)

PASS  sigma2 location: Python = 220.661   chapter = 220.66
PASS  SE location: Python = 186.485   chapter = 186.48
PASS  sigma2 rep(location): Python = 13.4463   chapter = 13.4463
PASS  SE rep(location): Python = 10.2484   chapter = 10.2484
PASS  sigma2 RIL: Python = 303.151   chapter = 303.15
PASS  SE RIL: Python = 57.509   chapter = 57.5088
PASS  sigma2 location*RIL: Python = 24.9323   chapter = 24.9323
PASS  SE location*RIL: Python = 6.67869   chapter = 6.6787
PASS  sigma2 Residual: Python = 64.8862   chapter = 64.8862
PASS  SE Residual: Python = 5.87452   chapter = 5.8745
PASS  AIC (SAS): Python = 3843.19   chapter = 3843.2
PASS  Intercept SE: Python = 7.87191   chapter = 7.8719


PROC MIXED gives no sums of squares, mean squares or F-tests for random terms, only variance-component estimates. With balanced data and all estimates positive, REML equals the method-of-moments estimates of notebook 01 (303.15, 24.93, 64.89). REML also keeps estimates non-negative, which the method of moments does not guarantee.

## Balanced data: mixed-model analysis with R (lme4)

**Original R code (from the chapter)**

```r
mm <- lmer(height ~ 1 + (1 | RIL) + (1 | location / rep) + (1 | location:RIL), data = maize)
summary(mm)
confint(mm)
```

lme4's estimates are the same; its AIC differs because it also counts the fixed intercept <sup>[N2]</sup>. Its "Std.Dev." column is $\sqrt{\hat\sigma^2}$, not a standard error. For uncertainty, `confint()` profiles the likelihood: for RIL the SD interval is (14.574, 21.155), i.e. (212.4, 447.5) on the variance scale, against the symmetric Wald interval 303.15 ± 2(57.51) = (188.1, 418.2). The profile interval is asymmetric, as the sampling distribution of a variance is.

`SparseREML.profile_ci()` computes profile intervals. lme4 profiles the ML deviance even for a REML fit, so an ML fit reproduces its numbers; a REML profile is also shown <sup>[N1]</sup>:

> **Note N2** (p. 13, fit statistics). AIC counts parameters differently: SAS counts the 5 variance parameters (3843.2); lme4 and pyreml also count the intercept (3845.2).
>
> **Note N1** (p. 14, lme4 confint()). lme4 profiles the ML deviance even for a REML fit. Its location interval (6.93, 32.40) matches an ML profile (6.89, 32.40); a REML profile gives (7.46, 44.54). The RIL interval is essentially unaffected.

In [6]:
ml_fit = model.fit(method="ml", se=None)
rows = []
for i, nm in enumerate(names + ["Residual"]):
    lo_ml, hi_ml = model.profile_ci(ml_fit, i)            # lme4-compatible
    lo_r, hi_r = model.profile_ci(fit, i)                 # REML profile
    rows.append((nm, lo_ml, hi_ml, lo_r, hi_r))
ci = pd.DataFrame(rows, columns=["term", "2.5% (lme4/ML)", "97.5% (lme4/ML)", "2.5% (REML)", "97.5% (REML)"]).set_index("term")
print("Profile 95% CIs on the SD scale\n", ci.round(3))

Profile 95% CIs on the SD scale
                2.5% (lme4/ML)  97.5% (lme4/ML)  2.5% (REML)  97.5% (REML)
term                                                                     
location                6.892           32.400        7.456        44.538
rep(location)           1.909            9.280        1.909         9.280
RIL                    14.574           21.155       14.584        21.181
location*RIL            3.571            6.281        3.571         6.281
Residual                7.390            8.826        7.390         8.826


In [7]:
lme4 = {"location*RIL": (3.571, 6.281), "RIL": (14.574, 21.155), "rep(location)": (1.909, 9.280), "location": (6.929, 32.399), "Residual": (7.390, 8.826)}
for nm, (lo, hi) in lme4.items():
    check(f"lme4 upper limit {nm}", ci.loc[nm, "97.5% (lme4/ML)"], hi, 0.02)
    check(f"lme4 lower limit {nm}", ci.loc[nm, "2.5% (lme4/ML)"], lo, 0.05)     # lme4 interpolates its profile with splines
print("RIL variance interval:", np.round(np.square(ci.loc["RIL", ["2.5% (lme4/ML)", "97.5% (lme4/ML)"]].to_numpy(float)), 1))

PASS  lme4 upper limit location*RIL: Python = 6.28139   chapter = 6.281
PASS  lme4 lower limit location*RIL: Python = 3.57087   chapter = 3.571
PASS  lme4 upper limit RIL: Python = 21.1546   chapter = 21.155
PASS  lme4 lower limit RIL: Python = 14.5742   chapter = 14.574
PASS  lme4 upper limit rep(location): Python = 9.27971   chapter = 9.28
PASS  lme4 lower limit rep(location): Python = 1.90865   chapter = 1.909
PASS  lme4 upper limit location: Python = 32.4   chapter = 32.399
PASS  lme4 lower limit location: Python = 6.8915   chapter = 6.929
PASS  lme4 upper limit Residual: Python = 8.82638   chapter = 8.826
PASS  lme4 lower limit Residual: Python = 7.39011   chapter = 7.39
RIL variance interval: [212.4 447.5]


### Likelihood-ratio test for the location variance

Drop location, refit, and compare $-2\log L_R$. The draft's lme4 code writes the reduced model's block term as `(1|rep/location)`, which nests location within rep and fits a different structure; the intended term is <mark>`(1|location:rep)`, which matches SAS's `random rep(location)`</mark><sup>[C3]</sup>.

> **Correction C3** (p. 15, LRT with lme4). Draft: reduced model (1|rep/location). Corrected: (1|location:rep). rep/location nests location within rep; the printed output shows rep and location:rep terms, i.e. a different model from SAS's random rep(location).

**Original R code (from the chapter)**

```r
mmR <- lmer(height ~ 1 + (1|RIL) + (1|rep/location) + (1|location:RIL), data = maize)
(LR <- mmR@devcomp$cmp["REML"] - mm@devcomp$cmp["REML"])
(p <- (1 / 2) * (1 - pchisq(LR, 1)))
```

In [8]:
dR = d.copy()
vcR = {"rep(location)": "0 + C(location):C(rep)", "RIL": "0 + C(RIL)", "location*RIL": "0 + C(RIL):C(location)"}
mmR = smf.mixedlm("height ~ 1", dR, groups="one", vc_formula=vcR).fit(reml=True, method=["bfgs"], gtol=1e-10, maxiter=5000)
D, p_raw, p_adj = boundary_lrt(-2 * mmR.llf, -2 * mm.llf)
print(f"-2RLL full {-2 * mm.llf:.2f}   reduced {-2 * mmR.llf:.2f}   LRT {D:.3f}   raw p {p_raw:.4f}   adjusted p {p_adj:.6f}")
check("-2RLL reduced (no location)", -2 * mmR.llf, 3841.3, 0.05)
check("LRT statistic", D, 8.145, 1e-3)
check("adjusted p-value", p_adj, 0.002159, 5e-6)

-2RLL full 3833.19   reduced 3841.34   LRT 8.145   raw p 0.0043   adjusted p 0.002159
PASS  -2RLL reduced (no location): Python = 3841.34   chapter = 3841.3
PASS  LRT statistic: Python = 8.14515   chapter = 8.145
PASS  adjusted p-value: Python = 0.00215883   chapter = 0.002159


The p-value is halved because the null hypothesis $\sigma^2_L = 0$ lies on the boundary of the parameter space (Self & Liang 1987). The adjusted p = 0.002 agrees with the synthesised ANOVA F-test for location (0.0022), whereas the Z-test (Z = 1.18, p = 0.12) misses it: the Z-test has low power for components estimated from few levels.

## Hypothesis testing with mixed models

The LRT statistic is $D = 2[\log L_1 - \log L_2]$, with $L_1$ the full and $L_2$ the reduced model's REML likelihood, compared with $\chi^2$ on the difference in the number of variance parameters (eq. 1.4). Information criteria (eqs. 1.5–1.6) are $\text{AIC} = -2\log L + 2t$ and $\text{BIC} = -2\log L + t\log\nu$. Two rules follow from REML's adjustment for the fixed effects:

| Fixed effects | Random effects | Estimation | AIC/BIC comparison |
|---|---|---|---|
| Same | Different | REML | Valid |
| Different | Same or different | REML | Not valid |
| Different | Same or different | ML | Valid |

The table of LRTs for the other components:

In [9]:
full = fit.neg2_loglik
rows = []
for drop, label in [(0, "No location"), (2, "No RIL"), (3, "No location*RIL")]:
    red = SparseREML(maize.height, np.ones((len(maize), 1)), [Z for i, Z in enumerate(Zs) if i != drop]).fit(se=None)
    D, p_raw, p_adj = boundary_lrt(red.neg2_loglik, full)
    rows.append((label, red.neg2_loglik, D, p_raw, p_adj))
lrt = pd.DataFrame(rows, columns=["Model", "-2 RLL", "LRT", "raw p", "adjusted p"]).set_index("Model")
print(f"Full model -2RLL = {full:.1f}"); print(lrt.round(4))
check("-2RLL no RIL", lrt.loc["No RIL", "-2 RLL"], 4092.7, 0.05)
check("LRT no RIL", lrt.loc["No RIL", "LRT"], 259.5, 0.05)
check("-2RLL no location*RIL", lrt.loc["No location*RIL", "-2 RLL"], 3850.4, 0.05)
check("LRT no location*RIL", lrt.loc["No location*RIL", "LRT"], 17.20, 0.05)

Full model -2RLL = 3833.2
                    -2 RLL       LRT   raw p  adjusted p
Model                                                   
No location      3841.3398    8.1451  0.0043      0.0022
No RIL           4092.7288  259.5341  0.0000      0.0000
No location*RIL  3850.4253   17.2306  0.0000      0.0000
PASS  -2RLL no RIL: Python = 4092.73   chapter = 4092.7
PASS  LRT no RIL: Python = 259.534   chapter = 259.5
PASS  -2RLL no location*RIL: Python = 3850.43   chapter = 3850.4
PASS  LRT no location*RIL: Python = 17.2306   chapter = 17.2


To compare models that differ in their **fixed** effects, refit by ML (`reml=False` in statsmodels, `method="ml"` in ASReml-R, `REML = FALSE` in lme4). The draft's lme4 snippet has <mark>`data = maize`</mark><sup>[C4]</sup> in place of `maize = df`. A Python sketch with two hypothetical covariates `x1`, `x2` (not in this data set):

```python
m1 = smf.mixedlm("y ~ x1", df, groups="RIL").fit(reml=False)
m2 = smf.mixedlm("y ~ x1 + x2", df, groups="RIL").fit(reml=False)
print(m1.aic, m2.aic, m1.bic, m2.bic)      # smaller is better
```

For the maize model, the ML fit and its information criteria:

> **Correction C4** (p. 19, ML comparison with lme4). Draft: lmer(y ~ x1 + x2 + (1|RIL), maize = df, ...). Corrected: lmer(..., data = maize, ...). `maize = df` is not an lmer argument.

In [10]:
print(f"ML: -2logL {ml_fit.neg2_loglik:.2f}  AIC {ml_fit.aic:.2f}  BIC {ml_fit.bic():.2f}")
print(f"REML: -2logL {fit.neg2_loglik:.2f}  AIC {fit.aic:.2f}  BIC (nu = n - p) {fit.bic():.2f}")
print("ML variance estimates:", dict(zip(names + ['Residual'], np.round(ml_fit.theta, 2))))

ML: -2logL 3839.03  AIC 3851.03  BIC 3876.27
REML: -2logL 3833.19  AIC 3843.19  BIC (nu = n - p) 3864.22
ML variance estimates: {'location': np.float64(167.98), 'rep(location)': np.float64(13.45), 'RIL': np.float64(302.61), 'location*RIL': np.float64(24.93), 'Residual': np.float64(64.89)}


## Prediction: BLUE and BLUP

LS-means of fixed effects are **best linear unbiased estimators (BLUEs)**: linear combinations of fixed-effect estimates, unbiased, with minimum variance. Random effects are *predicted* by **best linear unbiased predictors (BLUPs)**. In the mixed model a random factor's effects average to zero, so they drop out of fixed-effect LS-means, and an LS-mean of a random factor does not exist (PROC MIXED errors on `lsmeans RIL` when RIL is random). A marginal BLUP for a line is $\hat\mu + \hat u_k$. Animal breeders usually rank the $\hat u_k$ directly; crop breeders often use marginal predictions. A *conditional* BLUP, predicted for the set of tested environments, has a smaller standard error (Chapter 7).

**Original R code (from the chapter)**

```r
BLUP <- fixef(mm) + ranef(mm)$RIL
head(BLUP)
```

In [11]:
re = mm.random_effects[1]                               # one group, labelled by the value of `one`
u_sm = re[[i for i in re.index if i.startswith("RIL[C(RIL)")]]
u_sm.index = [i[len("RIL[C(RIL)["):-2] for i in u_sm.index]    # 'RIL[C(RIL)[RIL-1]]' -> 'RIL-1'
blup = pd.DataFrame({"Random effect predictor": u_sm, "mu": mm.fe_params.iloc[0]})
blup["BLUP"] = blup["mu"] + blup["Random effect predictor"]
lsmean = maize.groupby("RIL").height.mean()            # balanced: LS-mean = arithmetic mean of the line
blup["LSMEAN"] = lsmean
blup["Fixed effect estimate"] = lsmean - maize.height.mean()
s2 = sm_vc
h2 = s2["RIL"] / (s2["RIL"] + s2["location*RIL"] / 4 + s2["Residual"] / 8)
blup["Fixed effect*h2"] = blup["Fixed effect estimate"] * h2
show = ["RIL-1", "RIL-11", "RIL-12", "RIL-14", "RIL-15", "RIL-16", "RIL-20", "RIL-21"]
short_names = {"Random effect predictor": "u_hat", "Fixed effect estimate": "LSMEAN-mean", "Fixed effect*h2": "h2*(LSMEAN-mean)"}
print(blup.loc[show].rename(columns=short_names).round(2)); print(f"heritability of entry means h2 = {h2:.4f}")

        u_hat      mu    BLUP  LSMEAN  LSMEAN-mean  h2*(LSMEAN-mean)
RIL-1   -0.02  182.12  182.10  182.10        -0.02             -0.02
RIL-11   0.72  182.12  182.84  182.88         0.76              0.72
RIL-12   2.94  182.12  185.06  185.20         3.08              2.94
RIL-14  11.58  182.12  193.70  194.25        12.13             11.58
RIL-15  13.04  182.12  195.16  195.77        13.66             13.04
RIL-16  -8.87  182.12  173.24  172.82        -9.29             -8.87
RIL-20  26.38  182.12  208.50  209.75        27.63             26.38
RIL-21 -15.53  182.12  166.59  165.85       -16.27            -15.53
heritability of entry means h2 = 0.9548


In [12]:
for k, v in {"RIL-1": 182.10, "RIL-14": 193.70, "RIL-20": 208.50, "RIL-21": 166.59}.items():
    check(f"marginal BLUP {k}", blup.loc[k, "BLUP"], v, 6e-3)
check("BLUP = h2 x (LSmean - mean), max deviation", (blup["Random effect predictor"] - blup["Fixed effect*h2"]).abs().max(), 0, 1e-4)
check("heritability of entry means", h2, 0.9548, 5e-5)

PASS  marginal BLUP RIL-1: Python = 182.101   chapter = 182.1
PASS  marginal BLUP RIL-14: Python = 193.702   chapter = 193.7
PASS  marginal BLUP RIL-20: Python = 208.502   chapter = 208.5
PASS  marginal BLUP RIL-21: Python = 166.585   chapter = 166.59
PASS  BLUP = h2 x (LSmean - mean), max deviation: Python = 1.37845e-12   chapter = 0
PASS  heritability of entry means: Python = 0.954822   chapter = 0.9548


BLUPs are shrunk toward $\hat\mu = 182.12$ relative to the LS-means. RIL-20's LS-mean is 209.75 and its BLUP 208.50; RIL-21's LS-mean is 165.85 and its <mark>BLUP 166.59</mark><sup>[C5]</sup>. The amount of shrinkage depends on the heritability of entry means,

$$h^2 = \frac{\hat\sigma^2_G}{\hat\sigma^2_G + \hat\sigma^2_{GL}/n_l + \hat\sigma^2_\varepsilon/(n_l n_b)} = \frac{303.15}{303.15 + 24.93/4 + 64.89/8},$$

which equals <mark>0.955</mark><sup>[C23]</sup>. With balanced data each BLUP is exactly $h^2$ times the line's deviation from the grand mean (checked above).

> **Correction C5** (p. 22, BLUP vs LSmean text). Draft: RIL-21 … BLUP of 166.50. Corrected: BLUP of 166.59 (as in the table). Text disagrees with its own table.
>
> **Correction C23** (p. 23, heritability). Draft: 0.954. Corrected: 0.955 (0.9548). Rounding.